# RoBERTa + MLP sencillo para clasificación de décadas

Arquitectura mínima:

```text
texto -> tokenizer -> RoBERTa -> mean pooling -> MLP -> softmax 39 décadas
```

Este notebook genera:
- evaluación con split train/val/test
- matriz de confusión sin normalizar
- `submission_roberta_mlp_simple.csv` con el mejor modelo validado
- `submission_roberta_mlp_simple_full_data.csv` con modelo entrenado en todo `train.csv`

In [ ]:
# =========================
# 0. Instalación
# =========================
# Ejecuta esta celda una vez. Después reinicia el kernel.

%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
%pip install -U transformers accelerate sentencepiece protobuf tokenizers scikit-learn pandas numpy matplotlib tqdm scipy

In [ ]:
# =========================
# 1. Imports y configuración base
# =========================

import os
import gc
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

In [ ]:
# =========================
# 2. Hiperparámetros
# =========================

TRAIN_PATH = "train.csv"
EVAL_PATH = "eval.csv"

TEXT_COL = "text"
LABEL_COL = "decade"

MODEL_NAME = "Wariano/roberta-large-bne"

MAX_LEN = 512
BATCH_SIZE = 16       # súbelo a 32/64 si tu GPU aguanta
EPOCHS = 30
PATIENCE = 2

LR_ROBERTA = 1e-5
LR_MLP = 2e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.08
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.05

MLP_HIDDEN_1 = 1024
MLP_HIDDEN_2 = 512
MLP_HIDDEN_3 = 256
DROPOUT = 0.35

CHECKPOINT_DIR = "checkpoints_roberta_mlp_simple"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_roberta_mlp_simple.pt")
LAST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "last_roberta_mlp_simple.pt")

print("Modelo:", MODEL_NAME)
print("MAX_LEN:", MAX_LEN)
print("BATCH_SIZE:", BATCH_SIZE)

In [ ]:
# =========================
# 3. Cargar datos y etiquetas
# =========================

df = pd.read_csv(TRAIN_PATH)
assert TEXT_COL in df.columns
assert LABEL_COL in df.columns

df = df[[TEXT_COL, LABEL_COL]].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

decades = sorted(df[LABEL_COL].unique())
decade_to_idx = {decade: idx for idx, decade in enumerate(decades)}
idx_to_decade = {idx: decade for decade, idx in decade_to_idx.items()}
df["label_idx"] = df[LABEL_COL].map(decade_to_idx)
NUM_CLASSES = len(decades)

print("Shape:", df.shape)
print("NUM_CLASSES:", NUM_CLASSES)
print("Mapeo inicial:", list(decade_to_idx.items())[:5])
print("Mapeo final:", list(decade_to_idx.items())[-5:])
assert decade_to_idx[min(decades)] == 0
assert decade_to_idx[max(decades)] == NUM_CLASSES - 1
assert min(decades) == 150 and max(decades) == 188
display(df.head())

In [ ]:
# =========================
# 4. Split train/val/test
# =========================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_idx"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("Clase 150 train/val/test:", (train_df[LABEL_COL] == 150).sum(), (val_df[LABEL_COL] == 150).sum(), (test_df[LABEL_COL] == 150).sum())
print("Clase 188 train/val/test:", (train_df[LABEL_COL] == 188).sum(), (val_df[LABEL_COL] == 188).sum(), (test_df[LABEL_COL] == 188).sum())

In [ ]:
# =========================
# 5. Tokenizer
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)

sample = train_df.iloc[0][TEXT_COL]
enc = tokenizer(sample, max_length=MAX_LEN, truncation=True, padding="max_length", return_tensors="pt")
print(enc["input_ids"].shape, enc["attention_mask"].shape)
print(tokenizer.decode(enc["input_ids"][0][:80], skip_special_tokens=False))

In [ ]:
# =========================
# 6. Dataset y DataLoaders
# =========================

class RobertaDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe[TEXT_COL].astype(str).tolist()
        self.labels = dataframe["label_idx"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoded = tokenizer(
            text,
            max_length=MAX_LEN,
            truncation=True,
            padding="max_length",
            return_tensors=None
        )
        return {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def collate_fn(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "labels": torch.stack([x["label"] for x in batch]),
    }

train_loader = DataLoader(RobertaDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(RobertaDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(RobertaDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=torch.cuda.is_available())

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# =========================
# 7. Modelo RoBERTa + MLP simple
# =========================

class RobertaMLPClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden = self.roberta.config.hidden_size

        self.mlp = nn.Sequential(
            nn.Linear(hidden, MLP_HIDDEN_1),
            nn.LayerNorm(MLP_HIDDEN_1),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(MLP_HIDDEN_1, MLP_HIDDEN_2),
            nn.LayerNorm(MLP_HIDDEN_2),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(MLP_HIDDEN_2, MLP_HIDDEN_3),
            nn.LayerNorm(MLP_HIDDEN_3),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(MLP_HIDDEN_3, num_classes),
        )

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        logits = self.mlp(pooled)
        return logits

model = RobertaMLPClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)

try:
    model.roberta.gradient_checkpointing_enable()
    print("Gradient checkpointing activado.")
except Exception as e:
    print("No se pudo activar gradient checkpointing:", e)

if hasattr(model.roberta.config, "use_cache"):
    model.roberta.config.use_cache = False

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total params:", f"{total:,}")
print("Trainable params:", f"{trainable:,}")

In [ ]:
# =========================
# 8. Forward pass de prueba
# =========================

model.eval()
batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
        logits = model(input_ids, attention_mask)

print("input_ids:", input_ids.shape)
print("attention_mask sums:", attention_mask.sum(dim=1)[:10])
print("logits:", logits.shape)
print("labels min/max:", labels.min().item(), labels.max().item())
assert logits.shape == (labels.size(0), NUM_CLASSES)

In [ ]:
# =========================
# 9. Métricas, optimizer y scheduler
# =========================

idx_to_decade_array = np.array([idx_to_decade[i] for i in range(NUM_CLASSES)])

def compute_metrics(preds, labels):
    acc = accuracy_score(labels, preds)
    pred_decades = idx_to_decade_array[preds]
    true_decades = idx_to_decade_array[labels]
    abs_err = np.abs(pred_decades - true_decades)
    return {
        "acc": acc,
        "mae": abs_err.mean(),
        "acc_pm1": np.mean(abs_err <= 1),
        "acc_pm2": np.mean(abs_err <= 2),
    }

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

roberta_params = []
mlp_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("roberta."):
        roberta_params.append(param)
    else:
        mlp_params.append(param)

optimizer = torch.optim.AdamW([
    {"params": roberta_params, "lr": LR_ROBERTA, "weight_decay": WEIGHT_DECAY},
    {"params": mlp_params, "lr": LR_MLP, "weight_decay": WEIGHT_DECAY},
])

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))

print("Steps:", total_steps, "Warmup:", warmup_steps)
print("LR RoBERTa:", LR_ROBERTA, "LR MLP:", LR_MLP)

In [ ]:
# =========================
# 10. Train / evaluate
# =========================

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_examples = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_examples += bs
        preds = torch.argmax(logits.detach(), dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    metrics = compute_metrics(np.array(all_preds), np.array(all_labels))
    metrics["loss"] = total_loss / total_examples
    return metrics


@torch.no_grad()
def evaluate(model, loader, desc="Evaluating"):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc=desc, leave=False):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_examples += bs
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    metrics = compute_metrics(np.array(all_preds), np.array(all_labels))
    metrics["loss"] = total_loss / total_examples
    metrics["preds"] = np.array(all_preds)
    metrics["labels"] = np.array(all_labels)
    return metrics

In [ ]:
# =========================
# 11. Loop principal
# =========================

history = []
best_val_acc = -1.0
best_epoch = 0
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 70)

    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader, desc="Validation")

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_mae": train_metrics["mae"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_mae": val_metrics["mae"],
        "val_pm1": val_metrics["acc_pm1"],
        "val_pm2": val_metrics["acc_pm2"],
        "lr_roberta": optimizer.param_groups[0]["lr"],
        "lr_mlp": optimizer.param_groups[1]["lr"],
    }
    history.append(row)

    print(f"Train loss {row['train_loss']:.4f} | acc {row['train_acc']:.4f} | mae {row['train_mae']:.3f}")
    print(f"Val   loss {row['val_loss']:.4f} | acc {row['val_acc']:.4f} | mae {row['val_mae']:.3f} | ±1 {row['val_pm1']:.4f} | ±2 {row['val_pm2']:.4f}")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "best_val_acc": best_val_acc,
        "decade_to_idx": decade_to_idx,
        "idx_to_decade": idx_to_decade,
        "decades": decades,
        "config": {"MODEL_NAME": MODEL_NAME, "MAX_LEN": MAX_LEN, "NUM_CLASSES": NUM_CLASSES}
    }, LAST_MODEL_PATH)

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        best_epoch = epoch
        no_improve = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "best_val_acc": best_val_acc,
            "decade_to_idx": decade_to_idx,
            "idx_to_decade": idx_to_decade,
            "decades": decades,
            "config": {"MODEL_NAME": MODEL_NAME, "MAX_LEN": MAX_LEN, "NUM_CLASSES": NUM_CLASSES}
        }, BEST_MODEL_PATH)
        print("Nuevo mejor checkpoint guardado.")
    else:
        no_improve += 1
        print("Sin mejora:", no_improve)

    if no_improve >= PATIENCE:
        print("Early stopping activado.")
        break

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Mejor época:", best_epoch)
print("Mejor val acc:", best_val_acc)

In [ ]:
# =========================
# 12. Curvas
# =========================

history_df = pd.DataFrame(history)
display(history_df)

plt.figure(figsize=(10,4))
plt.plot(history_df["epoch"], history_df["train_acc"], label="train acc")
plt.plot(history_df["epoch"], history_df["val_acc"], label="val acc")
plt.legend(); plt.grid(True); plt.title("Accuracy"); plt.show()

plt.figure(figsize=(10,4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val loss")
plt.legend(); plt.grid(True); plt.title("Loss"); plt.show()

In [ ]:
# =========================
# 13. Evaluación test + matriz de confusión
# =========================

checkpoint = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Checkpoint:", BEST_MODEL_PATH)
print("Época:", checkpoint["epoch"])
print("Best val acc:", checkpoint["best_val_acc"])

test_metrics = evaluate(model, test_loader, desc="Test")
print("Test acc:", test_metrics["acc"])
print("Test mae:", test_metrics["mae"])
print("Test ±1:", test_metrics["acc_pm1"])
print("Test ±2:", test_metrics["acc_pm2"])

cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"], labels=list(range(NUM_CLASSES)))
cm_df = pd.DataFrame(cm, index=[idx_to_decade[i] for i in range(NUM_CLASSES)], columns=[idx_to_decade[i] for i in range(NUM_CLASSES)])
display(cm_df)

plt.figure(figsize=(15,12))
plt.imshow(cm, aspect="auto")
plt.title("Matriz de confusión sin normalizar - RoBERTa + MLP simple")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.xticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)], rotation=90)
plt.yticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)])
plt.colorbar(label="Conteo")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 14. Submission con mejor modelo validado
# =========================

class EvalRobertaDataset(Dataset):
    def __init__(self, dataframe):
        self.ids = dataframe["id"].tolist()
        self.texts = dataframe[TEXT_COL].fillna("").astype(str).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            truncation=True,
            padding="max_length",
            return_tensors=None
        )
        return {
            "id": self.ids[idx],
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
        }

def eval_collate_fn(batch):
    return {
        "ids": [x["id"] for x in batch],
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
    }

@torch.no_grad()
def predict_eval(model, eval_df, batch_size=BATCH_SIZE):
    loader = DataLoader(EvalRobertaDataset(eval_df), batch_size=batch_size, shuffle=False, collate_fn=eval_collate_fn, num_workers=0)
    all_ids, all_preds = [], []
    model.eval()
    for batch in tqdm(loader, desc="Predict eval"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy().tolist()
        all_ids.extend(batch["ids"])
        all_preds.extend(preds)
    answers = [idx_to_decade[int(i)] for i in all_preds]
    return pd.DataFrame({"id": all_ids, "answer": answers})

eval_df = pd.read_csv(EVAL_PATH)
assert "id" in eval_df.columns and TEXT_COL in eval_df.columns
eval_df[TEXT_COL] = eval_df[TEXT_COL].fillna("").astype(str)

submission = predict_eval(model, eval_df)
submission.to_csv("submission_roberta_mlp_simple.csv", index=False)
print("Guardado: submission_roberta_mlp_simple.csv")
display(submission.head())
display(submission["answer"].value_counts().sort_index())

In [ ]:
# =========================
# 15. Entrenamiento final con todo train.csv
# =========================

FINAL_EPOCHS = checkpoint.get("epoch", 5)
FINAL_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "final_roberta_mlp_simple_full_data.pt")

full_df = df.copy().reset_index(drop=True)
full_loader = DataLoader(RobertaDataset(full_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=torch.cuda.is_available())

# Liberar modelo anterior
model.to("cpu")
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_model = RobertaMLPClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)
try:
    final_model.roberta.gradient_checkpointing_enable()
except Exception:
    pass
if hasattr(final_model.roberta.config, "use_cache"):
    final_model.roberta.config.use_cache = False

roberta_params, mlp_params = [], []
for name, param in final_model.named_parameters():
    if name.startswith("roberta."):
        roberta_params.append(param)
    else:
        mlp_params.append(param)

final_optimizer = torch.optim.AdamW([
    {"params": roberta_params, "lr": LR_ROBERTA, "weight_decay": WEIGHT_DECAY},
    {"params": mlp_params, "lr": LR_MLP, "weight_decay": WEIGHT_DECAY},
])

final_steps = len(full_loader) * FINAL_EPOCHS
final_warmup = int(final_steps * WARMUP_RATIO)
final_scheduler = get_linear_schedule_with_warmup(final_optimizer, final_warmup, final_steps)
final_scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))

for epoch in range(1, FINAL_EPOCHS + 1):
    final_model.train()
    total_loss = 0.0
    total_examples = 0
    all_preds, all_labels = [], []
    print(f"\nFinal epoch {epoch}/{FINAL_EPOCHS}")
    for batch in tqdm(full_loader, desc="Final training", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        final_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = final_model(input_ids, attention_mask)
            loss = criterion(logits, labels)
        final_scaler.scale(loss).backward()
        final_scaler.unscale_(final_optimizer)
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), GRAD_CLIP)
        final_scaler.step(final_optimizer)
        final_scaler.update()
        final_scheduler.step()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_examples += bs
        all_preds.extend(torch.argmax(logits.detach(), dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())
    metrics = compute_metrics(np.array(all_preds), np.array(all_labels))
    print("loss:", total_loss / total_examples, "acc:", metrics["acc"], "mae:", metrics["mae"])

torch.save({
    "epoch": FINAL_EPOCHS,
    "model_state_dict": final_model.state_dict(),
    "decade_to_idx": decade_to_idx,
    "idx_to_decade": idx_to_decade,
    "decades": decades,
    "config": {"MODEL_NAME": MODEL_NAME, "MAX_LEN": MAX_LEN, "NUM_CLASSES": NUM_CLASSES}
}, FINAL_MODEL_PATH)
print("Modelo final guardado en:", FINAL_MODEL_PATH)

In [ ]:
# =========================
# 16. Submission final full-data
# =========================

final_model.eval()
submission_final = predict_eval(final_model, eval_df)
submission_final.to_csv("submission_roberta_mlp_simple_full_data.csv", index=False)
print("Guardado: submission_roberta_mlp_simple_full_data.csv")
display(submission_final.head())
display(submission_final["answer"].value_counts().sort_index())